**Revenue**

In [25]:
import requests
import csv
import time
from datetime import datetime
from pathlib import Path
import pandas as pd
from collections import defaultdict
REQUEST_SLEEP_SECONDS = 15

In [26]:
# get sp500 list
def get_sp500_tickers_alternative():

    # fetch SP500
    csv_url = 'https://raw.githubusercontent.com/datasets/s-and-p-500-companies/master/data/constituents.csv'
    df = pd.read_csv(csv_url)

    tickers = df['Symbol'].tolist()
    tickers = [ticker.replace('.', '-') for ticker in tickers]
    
    return tickers


sp500_symbols = get_sp500_tickers_alternative()

# validation
print("length of sp500_symbols:", len(sp500_symbols))
print(sp500_symbols[:10])

length of sp500_symbols: 503
['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A']


In [27]:
# API_KEY = "4WH041V75ZSYLZDY"  #ZYC's API key, 1
API_KEY = "4WH041V75ZSYLZDY"   # ZYC's API key, 2
#API_KEY = "5IOK22YFLKBUDHL"         # Jack's API key
# API_KEY = "LONB725JY0NFBGY6"          # WXY's
# API_KEY = "8TOEQYXQGWV486I1"    #XZJ

In [28]:
# change the index here
TODAY_TICKERS = sp500_symbols[500:]

In [29]:
def parse_fiscal_year(date_str):
    return int(date_str[:4])


def parse_quarterly_reports(quarterly_reports):
    """
    quarterly_reports: list from API
    return dict: {year: {"q1": v1, "q2": v2, "q3": v3, "q4": v4}}
    """
    year_quarters = defaultdict(list)

    # group by on fiscalDateEnding
    for rpt in quarterly_reports:
        year = parse_fiscal_year(rpt["fiscalDateEnding"])
        year_quarters[year].append(rpt)

    # get Q1 to Q4 for each year
    result = {}
    for year, rpts in year_quarters.items():
        rpts_sorted = sorted(rpts, key=lambda r: r["fiscalDateEnding"])

        # set None for some Q
        while len(rpts_sorted) < 4:
            rpts_sorted.append({"totalRevenue": None})

        # use totalRevenue
        q1 = rpts_sorted[0].get("totalRevenue")
        q2 = rpts_sorted[1].get("totalRevenue")
        q3 = rpts_sorted[2].get("totalRevenue")
        q4 = rpts_sorted[3].get("totalRevenue")

        result[year] = {
            "q1": q1,
            "q2": q2,
            "q3": q3,
            "q4": q4,
        }

    return result


def parse_annual_reports(annual_reports):
    """
    get the annual reports
    """
    result = {}
    for rpt in annual_reports:
        year = parse_fiscal_year(rpt["fiscalDateEnding"])
        result[year] = rpt.get("totalRevenue")
    return result


def get_company_financials(symbol, api_key):
    """
    get the information from the API
    """
    url = f"https://www.alphavantage.co/query?function=INCOME_STATEMENT&symbol={symbol}&apikey={api_key}"
    data = requests.get(url).json()

    if "quarterlyReports" not in data or "annualReports" not in data:
        print(f"API data missing for {symbol}, some error here, please check")
        return None

    # parse the reports
    q_dict = parse_quarterly_reports(data["quarterlyReports"])
    a_dict = parse_annual_reports(data["annualReports"])

    # merge the data for this company and year
    rows = []
    for year in sorted(q_dict.keys()):
        rows.append({
            "company_code": symbol,
            "year": year,
            "q1": q_dict[year]["q1"],
            "q2": q_dict[year]["q2"],
            "q3": q_dict[year]["q3"],
            "q4": q_dict[year]["q4"],
            "yearly_financial": a_dict.get(year)
        })

    return rows

In [30]:
# append the csv file
outfile = Path("company_income.csv")
all_rows = []

for ticker in TODAY_TICKERS:
    rows = get_company_financials(ticker, API_KEY)
    if rows:
        # only contains information after 2021
        rows = [r for r in rows if r["year"] >= 2021]
        all_rows.extend(rows)
    print(f"Sleeping {REQUEST_SLEEP_SECONDS} seconds to respect API rate limit...")
    time.sleep(REQUEST_SLEEP_SECONDS)

df = pd.DataFrame(all_rows)

if outfile.exists():
    old = pd.read_csv(outfile)
    df = pd.concat([old, df]).drop_duplicates(subset=["company_code", "year"])

df.to_csv(outfile, index=False)
print("Saved:", outfile)

Sleeping 15 seconds to respect API rate limit...
Sleeping 15 seconds to respect API rate limit...
Sleeping 15 seconds to respect API rate limit...
Saved: company_income.csv
